<table align="center">
 <td align="center"><a target="_blank" href="https://colab.research.google.com/github/ezponda/intro_deep_learning/blob/main/class/CNN/CLIP_Zero_Shot_Classification.ipynb">
        <img src="https://colab.research.google.com/img/colab_favicon_256px.png"  width="50" height="50" style="padding-bottom:5px;" />Run in Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/ezponda/intro_deep_learning/blob/main/class/CNN/CLIP_Zero_Shot_Classification.ipynb">
        <img src="https://github.githubassets.com/images/modules/logos_page/GitHub-Mark.png"  width="50" height="50" style="padding-bottom:5px;" />View Source on GitHub</a></td>
</table>

# Zero-Shot Image Classification with CLIP

## Table of Contents

1. [Introduction](#1.-Introduction)
2. [CLIP Architecture](#2.-CLIP-Architecture)
3. [Basic Zero-Shot Classification](#3.-Basic-Zero-Shot-Classification)
4. [Prompt Engineering](#4.-Prompt-Engineering)
5. [Image-Text Similarity Space](#5.-Image-Text-Similarity-Space)
6. [Zero-Shot on a Real Dataset](#6.-Zero-Shot-on-a-Real-Dataset)
7. [Beyond Object Categories](#7.-Beyond-Object-Categories)
8. [What's Next](#8.-What's-Next)

## 1. Introduction

In traditional image classification, you need to:
1. Collect thousands of labeled images
2. Train a model for hours
3. The model can only recognize the classes it was trained on

**Zero-shot classification** breaks this pattern: you describe categories in plain text, and the model classifies images immediately, without any training.

This is possible thanks to **CLIP** (Contrastive Language-Image Pre-training), a model by OpenAI trained on 400 million image-text pairs from the internet. CLIP learned to connect visual concepts with natural language, enabling it to understand almost any category you can describe in words.

## 2. CLIP Architecture

CLIP has two separate encoders trained together:

- **Image Encoder** (Vision Transformer): converts an image into a vector (embedding)
- **Text Encoder** (Transformer): converts a text description into a vector in the **same space**

During training, CLIP sees millions of (image, caption) pairs. It learns to place matching image-text pairs **close together** in the embedding space and non-matching pairs **far apart** (contrastive learning).

**For zero-shot classification:**
1. Encode the image into an embedding
2. Encode each candidate label (e.g., `"a photo of a dog"`) into text embeddings
3. Compute cosine similarity between the image and each text embedding
4. The highest similarity is the predicted class

Since CLIP was trained on internet-scale data, it understands an enormous range of visual concepts, far beyond any fixed set of categories.

In [ ]:
#!pip install transformers torch

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
import requests
from transformers import CLIPProcessor, CLIPModel

### Load the Model

We use `clip-vit-base-patch32`, a good balance between speed and accuracy (~150M parameters).

In [ ]:
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name)
processor = CLIPProcessor.from_pretrained(model_name)
print(f"Model loaded: {model_name}")

In [ ]:
IMG_BASE = "https://raw.githubusercontent.com/ezponda/intro_deep_learning/main/images"


def load_image(filename):
    """Load an image from the course repository."""
    url = f"{IMG_BASE}/{filename}"
    response = requests.get(url)
    return Image.open(BytesIO(response.content)).convert("RGB")


def classify_image(image, candidate_labels, template="a photo of a {}"):
    """Zero-shot classification with CLIP.

    Args:
        image: PIL Image
        candidate_labels: list of class names
        template: text template with {} placeholder for the label

    Returns:
        numpy array of probabilities for each label
    """
    texts = [template.format(label) for label in candidate_labels]
    inputs = processor(text=texts, images=image, return_tensors="pt", padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1).squeeze().numpy()
    return probs


def plot_classification(image, labels, probs, title="Zero-Shot Classification"):
    """Show image alongside classification probabilities."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4),
                                    gridspec_kw={"width_ratios": [1, 1.5]})
    ax1.imshow(image)
    ax1.axis("off")
    sorted_idx = np.argsort(probs)
    colors = ["#2196F3" if i != sorted_idx[-1] else "#4CAF50" for i in sorted_idx]
    ax2.barh([labels[i] for i in sorted_idx], [probs[i] for i in sorted_idx], color=colors)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel("Probability")
    ax2.set_title(title)
    plt.tight_layout()
    plt.show()

## 3. Basic Zero-Shot Classification

Let's classify images without any training. We just define candidate labels in text and CLIP does the rest.

In [ ]:
# Load a beach image from the course repository
image = load_image("beach.jpg")

# Define candidate labels
labels = ["beach", "mountain", "city", "forest", "desert"]
probs = classify_image(image, labels)

for label, prob in zip(labels, probs):
    print(f"  {label}: {prob:.1%}")

plot_classification(image, labels, probs, "Scene Classification")

In [ ]:
# Try with an animal image
image2 = load_image("white_shark.jpg")

labels2 = ["shark", "dolphin", "whale", "fish", "turtle"]
probs2 = classify_image(image2, labels2)

plot_classification(image2, labels2, probs2, "Marine Animal Classification")

### Open Vocabulary

Unlike traditional classifiers limited to their training classes, CLIP works with **any** text description. You can make labels as specific or creative as you want:

In [ ]:
image3 = load_image("dog_cat.jpg")

# Generic labels
generic_labels = ["animal", "vehicle", "building", "food", "landscape"]
probs_generic = classify_image(image3, generic_labels)
plot_classification(image3, generic_labels, probs_generic, "Generic Categories")

# Very specific descriptions - no template needed
specific_labels = ["a dog and a cat together", "two dogs playing",
                   "a cat sleeping alone", "a single dog"]
probs_specific = classify_image(image3, specific_labels, template="{}")
plot_classification(image3, specific_labels, probs_specific, "Specific Descriptions")

## 4. Prompt Engineering

The text you use to describe categories matters. CLIP was trained on internet captions, not single words. So `"a photo of a dog"` works better than just `"dog"`.

Common prompt templates:
- `"a photo of a {}"` - general purpose, good default
- `"a close-up photo of a {}"` - for detailed objects
- `"a photo of a {}, a type of pet"` - adding context helps
- `"a satellite image of {}"` - domain-specific context

In [ ]:
image = load_image("bird.jpg")

# Compare different templates
templates = [
    "{}",
    "a photo of a {}",
    "a close-up photo of a {}",
    "a photo of a {}, a type of animal",
    "a blurry photo of a {}",
]

all_labels = ["bird", "dog", "cat", "fish", "insect"]

print("Classifying with different templates:\n")
results = {}
for template in templates:
    probs = classify_image(image, all_labels, template=template)
    bird_prob = probs[0]  # bird is the first label
    template_display = template.format("...")
    results[template_display] = bird_prob
    print(f"  {template_display:<45s} bird: {bird_prob:.1%}")

# Plot comparison
plt.figure(figsize=(10, 4))
plt.barh(list(results.keys()), list(results.values()), color="#2196F3")
plt.xlabel("Probability assigned to 'bird'")
plt.title("Impact of Prompt Template on Classification")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## Question 1: Find the Best Prompt Template

Load the `traffic.jpg` image and classify it among `["traffic jam", "parking lot", "highway", "empty road", "accident"]`.

Try at least 3 different prompt templates and find which one gives the highest confidence for the correct class.

In [ ]:
image = load_image("traffic.jpg")

templates_to_try = [
    "a photo of {}",
    ...,
    ...,
]
labels = ["traffic jam", "parking lot", "highway", "empty road", "accident"]

for template in templates_to_try:
    probs = classify_image(image, labels, template=template)
    predicted = labels[np.argmax(probs)]
    confidence = np.max(probs)
    print(f"Template: {template.format('...'):<45s} -> {predicted} ({confidence:.1%})")

## 5. Image-Text Similarity Space

CLIP maps images and text to the **same** vector space. This means we can compute similarities between any combination of images and text descriptions.

Let's visualize this by building a similarity matrix between several images and text descriptions.

In [ ]:
# Load several diverse images
image_info = [
    ("beach.jpg", "Beach"),
    ("dog_cat.jpg", "Dog & Cat"),
    ("white_shark.jpg", "Shark"),
    ("traffic.jpg", "Traffic"),
    ("yoga_1.jpg", "Yoga"),
]
images = [load_image(f) for f, _ in image_info]
image_labels = [label for _, label in image_info]

# Define text descriptions
text_descriptions = [
    "a relaxing day at the beach",
    "cute pets playing together",
    "a dangerous ocean predator",
    "cars on a busy road",
    "a person doing exercise",
]

# Encode images
image_inputs = processor(images=images, return_tensors="pt", padding=True)
with torch.no_grad():
    image_features = model.get_image_features(**image_inputs)
    image_features = image_features / image_features.norm(dim=-1, keepdim=True)

# Encode texts
text_inputs = processor(text=text_descriptions, return_tensors="pt", padding=True)
with torch.no_grad():
    text_features = model.get_text_features(**text_inputs)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

# Similarity matrix: cosine similarity between all image-text pairs
similarity = (image_features @ text_features.T).numpy()

# Plot heatmap
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(similarity, cmap="Blues", vmin=0.15, vmax=0.35)
ax.set_xticks(range(len(text_descriptions)))
ax.set_xticklabels(text_descriptions, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(image_labels)))
ax.set_yticklabels(image_labels)

for i in range(len(image_labels)):
    for j in range(len(text_descriptions)):
        color = "white" if similarity[i, j] > 0.28 else "black"
        ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center",
                color=color, fontsize=11)

plt.colorbar(im, label="Cosine Similarity")
plt.title("Image-Text Similarity Matrix")
plt.tight_layout()
plt.show()

The diagonal has the highest values: each image matches best with its corresponding text description. This is the embedding space alignment that CLIP learned from 400 million image-text pairs.

## Question 2: Find the Closest Image

Write 3 creative text queries (e.g., `"a summer vacation"`, `"something scary"`, `"a healthy lifestyle"`). For each query, compute the similarity with all 5 images and find which one matches best.

Which queries give surprising results?

In [ ]:
queries = [
    ...,
    ...,
    ...,
]

for query in queries:
    text_input = processor(text=[query], return_tensors="pt", padding=True)
    with torch.no_grad():
        query_features = model.get_text_features(**text_input)
        query_features = query_features / query_features.norm(dim=-1, keepdim=True)

    similarities = (image_features @ query_features.T).squeeze().numpy()
    best_idx = np.argmax(similarities)
    print(f'Query: "{query}"')
    for label, sim in zip(image_labels, similarities):
        marker = " <-- best match" if label == image_labels[best_idx] else ""
        print(f"  {label:<12s} {sim:.3f} {'█' * int(sim * 50)}{marker}")
    print()

## 6. Zero-Shot on a Real Dataset

Let's evaluate CLIP on a real image classification dataset: **Oxford Flowers** (5 classes: daisy, dandelion, roses, sunflowers, tulips).

In the [Introduction to CNN](./Introduction_to_CNN.ipynb) notebook, you trained a CNN from scratch reaching ~72% accuracy, and with MobileNetV2 transfer learning reaching ~90%. Let's see how CLIP does **without any training at all**.

In [ ]:
import pathlib
import tarfile
import urllib.request
import os

# Download the same Flowers dataset used in Introduction_to_CNN
data_dir = pathlib.Path("flower_photos")
if not data_dir.exists():
    print("Downloading Flowers dataset...")
    url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
    urllib.request.urlretrieve(url, "flower_photos.tgz")
    with tarfile.open("flower_photos.tgz", "r:gz") as tar:
        tar.extractall()
    print("Done!")

class_names = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
print(f"Classes: {class_names}")

for cls in class_names:
    n = len(list((data_dir / cls).glob("*.jpg")))
    print(f"  {cls}: {n} images")

In [ ]:
# Evaluate CLIP zero-shot on a subset (30 images per class)
n_per_class = 30
template = "a photo of a {}, a type of flower"

correct = 0
total = 0
predictions = []
ground_truth = []

print(f"Evaluating CLIP with template: '{template}'")
print(f"Using {n_per_class} images per class...\n")

for cls in class_names:
    image_paths = sorted((data_dir / cls).glob("*.jpg"))[:n_per_class]
    cls_correct = 0
    for img_path in image_paths:
        img = Image.open(img_path).convert("RGB")
        probs = classify_image(img, class_names, template=template)
        predicted = class_names[np.argmax(probs)]
        predictions.append(predicted)
        ground_truth.append(cls)
        if predicted == cls:
            correct += 1
            cls_correct += 1
        total += 1
    print(f"  {cls}: {cls_correct}/{n_per_class} correct ({cls_correct/n_per_class:.0%})")

accuracy = correct / total
print(f"\nOverall accuracy: {accuracy:.1%} ({correct}/{total})")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(ground_truth, predictions, labels=class_names)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap="Blues", values_format="d")
plt.title(f"CLIP Zero-Shot Confusion Matrix (accuracy: {accuracy:.1%})")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print("Comparison:")
print(f"  CNN from scratch (Introduction_to_CNN):  ~72% val accuracy")
print(f"  MobileNetV2 transfer learning:           ~90% val accuracy")
print(f"  CLIP zero-shot (no training at all):      {accuracy:.1%}")

## Question 3: Improve CLIP Accuracy with Prompt Engineering

The accuracy above depends on the prompt template. Try to improve it by at least 5 percentage points using:

1. **Better templates**: e.g., `"a macro photo of a {}"`, `"a {} flower in a garden"`
2. **Ensemble**: average predictions from multiple templates
3. **Detailed class names**: e.g., `"yellow sunflower"` instead of `"sunflowers"`

In [ ]:
# Strategy 1: Try a better template
better_template = ...

# Strategy 2: Ensemble of templates
ensemble_templates = [
    ...,
    ...,
    ...,
]

# Evaluate with ensemble (average probabilities from all templates)
correct = 0
total = 0
for cls in class_names:
    image_paths = sorted((data_dir / cls).glob("*.jpg"))[:n_per_class]
    for img_path in image_paths:
        img = Image.open(img_path).convert("RGB")
        # Average probabilities from all templates
        all_probs = []
        for t in ensemble_templates:
            probs = classify_image(img, class_names, template=t)
            all_probs.append(probs)
        avg_probs = np.mean(all_probs, axis=0)
        predicted = class_names[np.argmax(avg_probs)]
        if predicted == cls:
            correct += 1
        total += 1

print(f"Ensemble accuracy: {correct/total:.1%}")

## 7. Beyond Object Categories

CLIP doesn't just recognize objects. It understands abstract concepts like mood, style, and quality. This enables classification tasks that would be impossible with traditional models.

In [ ]:
# Scene mood classification
image = load_image("beach.jpg")

mood_labels = ["calm and peaceful", "exciting and energetic",
               "scary and dangerous", "sad and melancholic"]
probs = classify_image(image, mood_labels, template="a {} scene")
plot_classification(image, mood_labels, probs, "Scene Mood")

In [ ]:
# Multi-attribute classification: extract several properties from one image
image = load_image("yoga_1.jpg")

attributes = {
    "Activity": (["yoga", "running", "swimming", "dancing", "resting"],
                 "a photo of a person doing {}"),
    "Setting": (["indoor", "outdoor", "at a gym", "at a studio", "at a park"],
                "a photo taken {}"),
    "Time of day": (["in the morning", "in the afternoon", "in the evening", "at night"],
                    "a photo taken {}"),
}

fig, axes = plt.subplots(1, len(attributes), figsize=(16, 4))
fig.suptitle("Multi-Attribute Classification", fontsize=14, y=1.02)

for ax, (attr_name, (attr_labels, template)) in zip(axes, attributes.items()):
    probs = classify_image(image, attr_labels, template=template)
    sorted_idx = np.argsort(probs)
    colors = ["#2196F3"] * len(attr_labels)
    colors[np.argmax(probs)] = "#4CAF50"
    sorted_colors = [colors[i] for i in sorted_idx]
    ax.barh([attr_labels[i] for i in sorted_idx],
            [probs[i] for i in sorted_idx], color=sorted_colors)
    ax.set_title(attr_name)
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.show()

## Question 4: Zero-Shot Domain Expert

CLIP was trained on internet data. How well does it understand specialized domains?

1. Load the chest X-ray image (`chest_radiograph.jpg`) and classify it among generic categories to verify CLIP recognizes it as a medical image
2. Try more specific medical classifications (e.g., `"healthy lungs"` vs `"pneumonia"` vs `"lung cancer"`)
3. Discuss: why might CLIP struggle with specialized medical images? What kind of data would it need to perform well?

In [ ]:
xray = load_image("chest_radiograph.jpg")

# Step 1: Generic classification
generic_labels = ["medical image", "photograph", "painting", "satellite image", "diagram"]
probs = classify_image(xray, generic_labels)
plot_classification(xray, generic_labels, probs, "Is this a medical image?")

# Step 2: More specific medical classification
medical_labels = [...]
probs_med = classify_image(xray, medical_labels, template=...)
plot_classification(xray, medical_labels, probs_med, "Medical Classification")

# Step 3: Discuss limitations
# - CLIP was trained on internet images, not specialized radiology datasets
# - Medical image interpretation requires years of specialized training
# - This shows the boundary of zero-shot: general knowledge vs. expert knowledge
# - Fine-tuning CLIP on medical data would dramatically improve performance

## 8. What's Next

In this notebook you used CLIP for **zero-shot classification**: assigning labels to images without any training. CLIP's versatility extends to many other tasks:

- **Image Search**: find images using text queries. See the [Image Search](../NLP/Image_search.ipynb) notebook
- **Open-Vocabulary Detection**: detect any object described in text. See the [Open Vocabulary Detection](./Open_Vocabulary_Detection.ipynb) notebook
- **Text-to-Image Generation**: CLIP guides diffusion models like Stable Diffusion

To go further:
- [OpenCLIP](https://github.com/mlfoundations/open_clip): open-source CLIP variants trained on LAION-2B (5x more data)
- [SigLIP](https://huggingface.co/google/siglip-base-patch16-224): Google's improved CLIP with sigmoid loss
- **Fine-tuning CLIP**: adapt CLIP to your specific domain for better accuracy on specialized tasks
- [CLIP Paper](https://arxiv.org/abs/2103.00020): *Learning Transferable Visual Models From Natural Language Supervision* (Radford et al., 2021)